<a href="https://colab.research.google.com/github/Qudus-O/Financial-News-Sentiment/blob/main/Financial_News_Sentiment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial News Sentiment: Fine-tuned DistilBERT vs. a TF-IDF Baseline

**Question:** How much does fine-tuning a pretrained transformer improve on a
classical bag-of-words baseline for classifying the sentiment of short financial
news posts, and where does it still fail?

## Summary
Fine-tuned `distilbert-base-uncased` reaches **0.835 macro-F1 (0.874 accuracy)** on a
held-out test set (mean of 3 seeds), against 0.690 macro-F1 for TF-IDF + logistic
regression. The biggest gain is on the Bearish class, where recall rises from 0.41
to 0.80, at the cost of more false alarms on Neutral tweets.

## Data
[`zeroshot/twitter-financial-news-sentiment`](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment):
short financial news tweets with a 3-class sentiment label.
- 9,543 training rows and 2,388 validation rows as published.
- License: MIT (dataset version 1.0.0, as stated on the dataset card).
- The tweets are loaded from the Hugging Face Hub at run time and are not copied
  into this repository.

| Label | Sentiment |
|---|---|
| 0 | Bearish |
| 1 | Bullish |
| 2 | Neutral |

The classes are imbalanced: Neutral is about 65% of the training data.

**Splits used here:**
- **Test set:** the published validation set (2,388 rows), not used for any
  training decision.
- **Dev set:** a stratified 10% of the training data (955 rows), used to monitor
  training only.
- **Fit set:** the remaining 8,588 training rows, used to train all models.

**Data quality:** after removing links and lowercasing, 38 of the 2,388 test texts
(1.6%) also appear in the training data, and 106 training rows duplicate other
training rows. I report the full test set, plus a check without the overlapping rows.

**A caveat on selection:** the choice of 3 epochs was informed by earlier exploratory
runs that scored this same test split, so the test scores carry a small selection
effect. Only the epoch count was chosen this way.

## Approach
1. Two baselines: a majority-class guess and TF-IDF + logistic regression.
2. Fine-tune `distilbert-base-uncased` (3 epochs, learning rate 2e-5, batch size 32,
   max length 64), repeated with 3 random seeds.
3. Report results on the test set, with per-class precision and recall.

## Metric
**Macro-F1** is the headline metric. Because Neutral dominates, accuracy alone is
misleading: a model that always predicts Neutral scores about 0.66 accuracy but
gets the other two classes completely wrong.

## Results
*Test set. DistilBERT is the mean of 3 seeds.*

| Model | Accuracy | Macro-F1 |
|---|---|---|
| Majority class | 0.656 | 0.264 |
| TF-IDF + logistic regression | 0.802 | 0.690 |
| DistilBERT (full fine-tune, 3 epochs) | 0.874 | 0.835 |

Seed-to-seed spread for DistilBERT is about 0.001 macro-F1 (3 seeds, so treat this
as approximate). Without the 38 overlapping test rows, DistilBERT scores 0.834
macro-F1 and 0.873 accuracy.

## Main findings
- Fine-tuning improves macro-F1 by about 0.145 over the TF-IDF baseline, far more
  than the seed-to-seed spread.
- Bearish recall rises from 0.41 to 0.80 (seed-to-seed std 0.014), and Bullish recall
  from 0.56 to 0.80. The cost: about 9% of Neutral tweets are flagged as Bearish or
  Bullish, up from about 4%.
- Remaining Bearish misses are often events with no negative word (a probe, a data
  breach), and some "false alarms" are tweets with mixed signals or arguable labels
  (based on a random sample of 8 per group, so a description of that sample and not
  a general claim).
- Duplicate texts between train and test have a negligible effect (about 0.001
  macro-F1).


## How to run
Open in Google Colab with a GPU runtime (Runtime → Change runtime type → T4),
then Runtime → Run all. The full run takes a few minutes. Library versions are
printed in the Setup section.

## Setup
Library versions are printed so the run can be reproduced. All randomness is
seeded (seed 42 here; seeds 0-2 for the fine-tuning runs).

In [1]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib

import re
import numpy as np
import torch, sklearn, transformers, datasets
from transformers import set_seed

set_seed(42)
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| sklearn", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

torch 2.11.0+cu130 | transformers 5.18.0 | datasets 4.8.5 | sklearn 1.6.1
GPU available: True


## Data
Three classes: 0 = Bearish, 1 = Bullish, 2 = Neutral. Neutral is about 65% of
the training data. The file is not shuffled (the first rows contain no neutral
tweets), so examples below are sampled randomly.

In [2]:
from collections import Counter
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
LABELS = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

print(ds)
print({LABELS[k]: v for k, v in sorted(Counter(ds["train"]["label"]).items())})

shuf = ds["train"].shuffle(seed=42).select(range(300))
for lab, name in LABELS.items():
    print(name)
    for t in [r["text"] for r in shuf if r["label"] == lab][:2]:
        print("  -", t)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'Bearish': 1442, 'Bullish': 1923, 'Neutral': 6178}
Bearish
  - Hong Kong stocks slip as whistle-blower doctor dies of virus
  - STOCKS AT THE CLOSE: 
- Dow down down 0.09% 
- Nasdaq down 0.11% 
- S&P down 0.07% https://t.co/7L7WyRWDsE
Bullish
  - Leafly Jobs Report: Cannabis is the fastest-growing American industry, surpassing 240,000 jobs
  - Added $STNE , $GSX and $BKE to the #PowerEarningsGap list 

These 3 names look very promising going forward! Look f… https://t.co/yABsTNVDIO
Neutral
  - Stocks - Tiffany, Disney, Tesla Rise Premarket; Uber Falls
  - Digital Realty Trust declares $1.08 dividend


In [3]:
norm = lambda t: re.sub(r"https?://\S+", "", t).strip().lower()
train_texts = {norm(t) for t in ds["train"]["text"]}
dup = [t for t in ds["validation"]["text"] if norm(t) in train_texts]

print(len(dup), "of", len(ds["validation"]), "validation texts also appear in train")
print(dup[:3])
print(len(ds["train"]), "train rows,", len(train_texts), "unique after normalizing")

38 of 2388 validation texts also appear in train
['Fed Chairman Jerome Powell puts lawmakers on notice that fiscal policy may need to play a bigger role countering do… https://t.co/URFHNRkmfh', 'Exxon is cutting billions of dollars in investments to keep its financial footing during the worst oil-price rout i… https://t.co/MI5vCEkhlm', '3M is exploring a sale of its drug delivery systems unit, which could fetch about $1 billion https://t.co/ISlbwO3FCd']
9543 train rows, 9437 unique after normalizing


### Data quality check
After removing links and lowercasing, 38 of the 2,388 test texts (1.6%) also
appear in the training data, and 106 training rows duplicate other training rows.
The cell below checks whether any duplicates carry conflicting labels and how
many dev rows also appear in the fit split. Results are reported on the full test
set and, as a sensitivity check, on the test set without the overlapping rows.

In [4]:
raw_train = ds["train"].class_encode_column("label")
parts = raw_train.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
fit_raw, dev_raw = parts["train"], parts["test"]
test_raw = ds["validation"]
print(len(fit_raw), len(dev_raw), len(test_raw))

8588 955 2388


### Train / dev / test split
The published validation set is used as the **test set** and is not used for any
decision. A stratified 10% of the training data (955 rows) is held out as a **dev
set** to monitor training. The other 8,588 rows are used to fit the models.
Stratifying keeps the class proportions the same in each part. Split seed: 42.

In [5]:
train_labels = {}
for t, l in zip(ds["train"]["text"], ds["train"]["label"]):
    train_labels.setdefault(norm(t), set()).add(l)

test_dup_mask = np.array([norm(t) in train_labels for t in test_raw["text"]])
test_conflicts = sum(
    1 for t, l, d in zip(test_raw["text"], test_raw["label"], test_dup_mask)
    if d and l not in train_labels[norm(t)]
)
fit_norm = {norm(t) for t in fit_raw["text"]}

print("overlapping test rows:", int(test_dup_mask.sum()))
print("of those, label differs from every train label for that text:", test_conflicts)
print("train texts with more than one label:", sum(len(v) > 1 for v in train_labels.values()))
print("dev rows also in fit:", sum(norm(t) in fit_norm for t in dev_raw["text"]), "of", len(dev_raw))

overlapping test rows: 38
of those, label differs from every train label for that text: 1
train texts with more than one label: 2
dev rows also in fit: 16 of 955


**What this shows:** Of the 38 overlapping test rows, 37 carry a label that also
appears for the same text in the training data; one has a different label. Only 2
training texts have more than one label, so labeling of exact duplicates is almost
fully consistent. This check covers exact duplicates only and says nothing about
label noise elsewhere. 16 of the 955 dev rows (1.7%) also appear in the fit split,
which makes dev scores slightly optimistic; dev is used only to monitor training.

## Baselines
Before training a transformer, I need numbers it has to beat. Both baselines use
only the fit split for training and are scored on the held-out test set.

- **Majority class:** always predicts the most common training label (neutral).
- **TF-IDF + logistic regression:** unigram and bigram TF-IDF features, with the
  vectorizer fitted on training text only so nothing leaks from the test set.

The baseline settings were not tuned, so a tuned baseline would likely score
somewhat higher. That makes the comparison slightly favorable to the transformer.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_fit, y_test = fit_raw["label"], test_raw["label"]

majority = Counter(y_fit).most_common(1)[0][0]
mj_pred = [majority] * len(y_test)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_fit = vec.fit_transform(fit_raw["text"])
clf = LogisticRegression(max_iter=1000).fit(X_fit, y_fit)
tfidf_pred = clf.predict(vec.transform(test_raw["text"]))

for name, p in [("Majority class", mj_pred), ("TF-IDF + LR", tfidf_pred)]:
    print(name, "acc", round(accuracy_score(y_test, p), 4),
          "macro-F1", round(f1_score(y_test, p, average="macro"), 4))
print(classification_report(y_test, tfidf_pred, digits=3))

Majority class acc 0.6558 macro-F1 0.264
TF-IDF + LR acc 0.8023 macro-F1 0.6903
              precision    recall  f1-score   support

           0      0.797     0.406     0.538       347
           1      0.796     0.558     0.656       475
           2      0.804     0.964     0.877      1566

    accuracy                          0.802      2388
   macro avg      0.799     0.643     0.690      2388
weighted avg      0.801     0.802     0.784      2388



**What this shows:**
- The majority guess reaches 0.656 accuracy but only 0.264 macro-F1, because it
  never predicts Bearish or Bullish. That is why macro-F1 is the headline metric.
- TF-IDF + logistic regression reaches 0.802 accuracy and 0.690 macro-F1.
  Precision is about 0.80 for every class, but recall differs sharply: it finds
  only 41% of Bearish and 56% of Bullish tweets, against 96% of Neutral. It
  defaults to Neutral when unsure, so most bearish news is missed.

## Tokenization
`distilbert-base-uncased` tokenizer. Length statistics are computed on the fit
split only. `max_length=64` is chosen so that only a small share of tweets is
truncated (the share is printed below). Padding is done per batch by the data
collator, not here, to avoid wasting compute on pad tokens.

In [7]:
from transformers import AutoTokenizer

MODEL = "distilbert-base-uncased"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL)

lengths = np.array([len(tokenizer(t)["input_ids"]) for t in fit_raw["text"]])
print("token length percentiles (50/90/99/max):", np.percentile(lengths, [50, 90, 99, 100]))
print("share truncated at", MAX_LEN, ":", round(float((lengths > MAX_LEN).mean()), 4))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

fit_ds = fit_raw.map(tokenize, batched=True)
dev_ds = dev_raw.map(tokenize, batched=True)
test_ds = test_raw.map(tokenize, batched=True)
print(fit_ds)

token length percentiles (50/90/99/max): [25. 47. 57. 82.]
share truncated at 64 : 0.0022
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8588
})


**What this shows:** Median tweet length is 25 tokens, the 99th percentile is 57,
and the longest is 82. With `max_length=64`, only 0.22% of fit tweets (roughly 19
of 8,588) are truncated, so truncation has a negligible effect.

## Fine-tuning
Full fine-tuning of `distilbert-base-uncased` with a new 3-class classification head.

- 3 epochs, learning rate 2e-5, batch size 32, max length 64, fp16.
- Trained on the fit split; the dev split is used to monitor training only.
- Repeated with 3 seeds (0, 1, 2). Test-set predictions from each run are saved
  so that per-class results and errors come from the final models.

In [8]:
import pandas as pd
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, set_seed)
from sklearn.metrics import accuracy_score, f1_score

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

SEEDS = [0, 1, 2]
test_preds, rows = {}, []

for seed in SEEDS:
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    args = TrainingArguments(
        output_dir=f"ft-seed{seed}", num_train_epochs=3,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
        fp16=True, report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=fit_ds, eval_dataset=dev_ds,
                      data_collator=collator, processing_class=tokenizer,
                      compute_metrics=compute_metrics)
    trainer.train()
    dev = trainer.evaluate()
    out = trainer.predict(test_ds)
    test_preds[seed] = np.argmax(out.predictions, axis=-1)
    rows.append({"seed": seed, "dev_macro_f1": dev["eval_macro_f1"],
                 "test_acc": out.metrics["test_accuracy"],
                 "test_macro_f1": out.metrics["test_macro_f1"]})
    print(rows[-1])
    if seed == SEEDS[0]:
        final_trainer = trainer

res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("test acc      mean", round(res.test_acc.mean(), 4), "std", round(res.test_acc.std(ddof=0), 4))
print("test macro-F1 mean", round(res.test_macro_f1.mean(), 4), "std", round(res.test_macro_f1.std(ddof=0), 4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.445125,0.832461,0.773828
2,0.474417,0.395678,0.851309,0.810581
3,0.474417,0.401673,0.851309,0.806136


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.474417,0.401673,3,0.851309,0.806136


{'seed': 0, 'dev_macro_f1': 0.8061359705113501, 'test_acc': 0.8747906197654941, 'test_macro_f1': 0.8363960113960114}


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.444219,0.826178,0.757709
2,0.494433,0.403501,0.854450,0.809537
3,0.494433,0.412637,0.859686,0.814509


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.494433,0.412637,3,0.859686,0.814509


{'seed': 2, 'dev_macro_f1': 0.814508632012684, 'test_acc': 0.8726968174204355, 'test_macro_f1': 0.8335414812921744}
 seed  dev_macro_f1  test_acc  test_macro_f1
    0        0.8061    0.8748         0.8364
    1        0.8165    0.8744         0.8358
    2        0.8145    0.8727         0.8335
test acc      mean 0.874 std 0.0009
test macro-F1 mean 0.8352 std 0.0012


**What this shows:** Across three seeds, DistilBERT reaches 0.874 test accuracy and
0.835 test macro-F1 (seed-to-seed std about 0.001; with only 3 seeds, treat that as
approximate). Dev macro-F1 is lower (0.806 to 0.816), but dev is only 955 rows with
few Bearish examples, so it is a noisy estimate. It was used for monitoring only.

Per-class precision, recall and F1 are computed per seed and averaged. The
sensitivity check removes the 38 test rows that also appear in the training data.

In [9]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

y = np.array(test_raw["label"])
names = [LABELS[i] for i in range(3)]

per = np.array([
    np.stack(precision_recall_fscore_support(y, test_preds[s], labels=[0, 1, 2], zero_division=0)[:3])
    for s in SEEDS])                                   # shape: seeds x (P, R, F1) x classes
tf = precision_recall_fscore_support(y, tfidf_pred, labels=[0, 1, 2], zero_division=0)

tbl = pd.DataFrame({
    "class": names,
    "precision": per[:, 0].mean(0), "recall": per[:, 1].mean(0), "f1": per[:, 2].mean(0),
    "recall_std": per[:, 1].std(0),
    "tfidf_recall": tf[1], "tfidf_f1": tf[2]})
print(tbl.round(3).to_string(index=False))

cm = np.mean([confusion_matrix(y, test_preds[s], labels=[0, 1, 2]) for s in SEEDS], axis=0)
print(pd.DataFrame(cm.round(1), index=[f"true {n}" for n in names],
                   columns=[f"pred {n}" for n in names]))

keep = ~test_dup_mask
print("test rows kept:", int(keep.sum()))
bert_acc = np.mean([accuracy_score(y[keep], test_preds[s][keep]) for s in SEEDS])
bert_f1 = np.mean([f1_score(y[keep], test_preds[s][keep], average="macro") for s in SEEDS])
print("DistilBERT without overlap: acc", round(bert_acc, 4), "macro-F1", round(bert_f1, 4))
print("TF-IDF without overlap:     acc", round(accuracy_score(y[keep], tfidf_pred[keep]), 4),
      "macro-F1", round(f1_score(y[keep], tfidf_pred[keep], average="macro"), 4))

  class  precision  recall    f1  recall_std  tfidf_recall  tfidf_f1
Bearish      0.767   0.804 0.785       0.014         0.406     0.538
Bullish      0.816   0.796 0.806       0.011         0.558     0.656
Neutral      0.916   0.913 0.914       0.002         0.964     0.877
              pred Bearish  pred Bullish  pred Neutral
true Bearish         279.0          18.0          50.0
true Bullish          15.7         378.3          81.0
true Neutral          69.0          67.3        1429.7
test rows kept: 2350
DistilBERT without overlap: acc 0.8729 macro-F1 0.8343
TF-IDF without overlap:     acc 0.8013 macro-F1 0.6902


**What this shows (test set, mean of 3 seeds):**
- Bearish recall rises from 0.41 (TF-IDF) to 0.80 (seed-to-seed std 0.014), and
  Bullish recall from 0.56 to 0.80. Bearish precision is 0.77 versus 0.80 for TF-IDF.
- The cost is on Neutral: recall falls from 0.96 to 0.91, so about 8.7% of neutral
  tweets are flagged as Bearish or Bullish, up from about 3.6%.
- Without the 38 test rows that also appear in training, macro-F1 is 0.834 instead
  of 0.835 and accuracy 0.873 instead of 0.874, so duplication has a negligible effect.

## Error analysis
Randomly sampled errors from the seed-0 model on the test set: Bearish tweets
predicted Neutral (missed bearish news) and Neutral tweets predicted Bearish
(false alarms). A random sample avoids cherry-picking.

In [10]:
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame({"text": test_raw["text"], "true": y, "pred": test_preds[0]})
df["true"] = df["true"].map(LABELS)
df["pred"] = df["pred"].map(LABELS)

missed = df[(df.true == "Bearish") & (df.pred == "Neutral")]
alarm = df[(df.true == "Neutral") & (df.pred == "Bearish")]

print(len(missed), "Bearish tweets predicted Neutral (seed 0)")
print(missed.sample(8, random_state=0).to_string())
print()
print(len(alarm), "Neutral tweets predicted Bearish (seed 0)")
print(alarm.sample(8, random_state=0).to_string())

46 Bearish tweets predicted Neutral (seed 0)
                                                                                                                   text     true     pred
966          Tech War Set To Deepen As China Stockpiles US Chips Amid 'Silicon Curtain' Threat  https://t.co/sMw7Z0Xo3w  Bearish  Neutral
959                                                       Every Bubble Eventually Finds Its Pin https://t.co/E3LesbBMP1  Bearish  Neutral
1451                                                                      EU probes Qualcomm over radio-frequency chips  Bearish  Neutral
197                                                    Mahindra Says Coronavirus May Jeopardize India Emissions Rollout  Bearish  Neutral
505   T-Mobile said a hacker gained access to limited information contained in prepaid accounts https://t.co/Ui6KpaVAmj  Bearish  Neutral
1460                                               UPDATE 1-Italian police search ArcelorMittal offices over Ilva probe  Bearis

**What this shows (8 randomly sampled errors per group, seed-0 model, so a
description of this sample and not a general claim):**
- Of 8 missed Bearish tweets, 4 describe an event (probe, breach, police search,
  a bubble metaphor) with no negative word, 2 contain negative words in a hedged or
  sector-wide context, and 2 have labels that are arguable.
- Of 8 Neutral tweets flagged as Bearish, 3 carry mixed signals (earnings beat on one
  measure, miss on another), 3 have Neutral labels that look arguable, and 2 are
  plausibly neutral but contain negative keywords.
- Some of the apparent errors may therefore reflect label ambiguity, not only model
  weakness. I did not check how the labels were assigned.

## Limitations and next steps
**Limitations**
- One dataset: short English financial tweets with three classes. Results may not
  transfer to longer documents or other sources.
- The Bearish/Neutral boundary is ambiguous in places, and I did not check the
  labeling process, so some of the measured error is likely label noise.
- Three seeds and one train/dev split: the reported spread is approximate.
- The 3-epoch choice was informed by earlier exploratory runs that scored this test
  split, so there is a small selection effect.
- The baselines were not tuned.
- Costs of a missed bearish tweet versus a false alarm depend on the user; I report
  both, but did not optimize for either.

**Next steps**
- Compare with a finance-pretrained model such as FinBERT.
- Fine-tune with LoRA and compare quality against the full fine-tune.
- Tune the decision threshold for Bearish to trade recall against false alarms.